In [3]:
import pandas as pd
import numpy as np

# Base paths
MAPPING = "../Progress/code/datasets/mapping"
OG      = "../Progress/code/datasets/og"

# Load all datasets
occ_base     = pd.read_csv(f"{MAPPING}/occupation_base_soc.csv")
skill_bridge = pd.read_csv(f"{MAPPING}/occupation_skill_bridge_onet.csv", low_memory=False)
tech_summary = pd.read_csv(f"{MAPPING}/occupation_technology_skills_onet_summary.csv")
best_match   = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_best_thr30.csv")
candidates   = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_candidates_thr30_long.csv")

salary       = pd.read_excel(f"{OG}/occupation-salary.xlsx")
automation   = pd.read_csv(f"{OG}/automation-data-by-state.csv", encoding="latin-1")  # fixes unicode error

print("All files loaded.")
print(f"  occ_base:     {occ_base.shape}")
print(f"  skill_bridge: {skill_bridge.shape}")
print(f"  tech_summary: {tech_summary.shape}")
print(f"  best_match:   {best_match.shape}")
print(f"  candidates:   {candidates.shape}")
print(f"  salary:       {salary.shape}")
print(f"  automation:   {automation.shape}")

All files loaded.
  occ_base:     (702, 4)
  skill_bridge: (94682, 13)
  tech_summary: (618, 6)
  best_match:   (57085, 8)
  candidates:   (4357770, 8)
  salary:       (1394, 20)
  automation:   (702, 54)


In [ ]:
# Keep only detailed-level rows from salary (avoid double-counting broad/minor groups)
salary_detail = salary[salary["OCC_GROUP"] == "detailed"].copy()

# Normalize SOC code to match SOC6 format (already XX-XXXX in both)
salary_detail["SOC6"] = salary_detail["OCC_CODE"].astype(str).str.strip()
occ_base["SOC6"]      = occ_base["SOC6"].astype(str).str.strip()

# Select only what we need for GCSI
salary_cols = salary_detail[["SOC6", "TOT_EMP", "A_MEDIAN", "H_PCT10", "H_PCT90"]].copy()

# Replace * (suppressed values) with NaN
salary_cols.replace("*", np.nan, inplace=True)
salary_cols["TOT_EMP"]  = pd.to_numeric(salary_cols["TOT_EMP"],  errors="coerce")
salary_cols["A_MEDIAN"] = pd.to_numeric(salary_cols["A_MEDIAN"], errors="coerce")
salary_cols["H_PCT10"]  = pd.to_numeric(salary_cols["H_PCT10"],  errors="coerce")
salary_cols["H_PCT90"]  = pd.to_numeric(salary_cols["H_PCT90"],  errors="coerce")

# Join
occ_enriched = occ_base.merge(salary_cols, on="SOC6", how="left")

print(f"Enriched occupation table: {occ_enriched.shape}")
print(f"Rows with salary data:     {occ_enriched['A_MEDIAN'].notna().sum()}")
print(f"Rows missing salary:       {occ_enriched['A_MEDIAN'].isna().sum()}")

Enriched occupation table: (702, 8)
Rows with salary data:     679
Rows missing salary:       23


In [5]:
df = occ_enriched.copy()

# Wage stability = lower spread is MORE stable, so we invert it
df["wage_spread"] = (df["H_PCT90"] - df["H_PCT10"]).clip(lower=0)

# Min-max normalizer
def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# Normalize each component
df["income_strength"]   = minmax(df["A_MEDIAN"].fillna(df["A_MEDIAN"].median()))
df["employment_scale"]  = minmax(df["TOT_EMP"].fillna(df["TOT_EMP"].median()))
df["wage_stability"]    = 1 - minmax(df["wage_spread"].fillna(df["wage_spread"].median()))
df["automation_safety"] = 1 - df["automation_probability"].fillna(df["automation_probability"].median())

# GCSI formula from your proposal
df["GCSI"] = (
    0.35 * df["income_strength"] +
    0.25 * df["employment_scale"] +
    0.20 * df["wage_stability"] +
    0.20 * df["automation_safety"]
) * 100  # scale to 0-100

# Round to 2 decimal places
df["GCSI"] = df["GCSI"].round(2)

# Save
gcsi_cols = ["SOC6", "occupation_name", "onet_title",
             "automation_probability", "A_MEDIAN", "TOT_EMP",
             "income_strength", "employment_scale",
             "wage_stability", "automation_safety", "GCSI"]

occupation_gcsi = df[gcsi_cols].copy()
occupation_gcsi.to_csv(f"{MAPPING}/occupation_gcsi.csv", index=False)

print(f"occupation_gcsi.csv saved: {occupation_gcsi.shape}")
print(occupation_gcsi[["occupation_name", "GCSI"]].sort_values("GCSI", ascending=False).head(10))

occupation_gcsi.csv saved: (702, 11)
                               occupation_name   GCSI
0                             Chief Executives  71.41
232                          Dentists; General  65.07
7    Computer and Information Systems Managers  61.91
1              General and Operations Managers  61.66
3                           Marketing Managers  60.52
165                                    Lawyers  59.57
8                           Financial Managers  59.25
4                               Sales Managers  58.56
241                                Podiatrists  58.25
235                            Prosthodontists  57.46


In [6]:
# Normalize join key — both to lowercase stripped string
occupation_gcsi["occupation_name_key"] = occupation_gcsi["occupation_name"].str.lower().str.strip()
best_match["career_match_key"]         = best_match["career_match"].str.lower().str.strip()

best_enriched = best_match.merge(
    occupation_gcsi[["occupation_name_key", "SOC6", "GCSI",
                      "automation_probability", "A_MEDIAN", "TOT_EMP"]],
    left_on="career_match_key",
    right_on="occupation_name_key",
    how="left"
)

# Hybrid score: 50% semantic similarity + 50% GCSI (normalized to 0-1)
best_enriched["hybrid_score"] = (
    0.50 * best_enriched["similarity"] +
    0.50 * (best_enriched["GCSI"] / 100)
).round(4)

best_enriched.drop(columns=["occupation_name_key", "career_match_key"], inplace=True)
best_enriched.to_csv(f"{MAPPING}/program_best_enriched_gcsi.csv", index=False)

print(f"program_best_enriched_gcsi.csv saved: {best_enriched.shape}")
print(f"Rows with GCSI matched: {best_enriched['GCSI'].notna().sum()}")
print(f"Rows missing GCSI:      {best_enriched['GCSI'].isna().sum()}")

program_best_enriched_gcsi.csv saved: (57085, 14)
Rows with GCSI matched: 34622
Rows missing GCSI:      22463


In [8]:
unmatched = best_enriched[best_enriched["GCSI"].isna()]["career_match"].dropna().str.lower().str.strip().unique()
matched_names = occupation_gcsi["occupation_name"].dropna().str.lower().str.strip().unique()

print(f"Unmatched career names: {len(unmatched)}")
print("\nSample unmatched:")
for name in sorted(unmatched)[:20]:
    print(f"  '{name}'")

print("\nSample matched occupation names from GCSI table:")
for name in sorted(matched_names)[:20]:
    print(f"  '{name}'")

Unmatched career names: 419

Sample unmatched:
  'actors producers and directors'
  'administrative law judges adjudicators and hearing officers'
  'advertising marketing promotions public relations and sales managers'
  'agents and business managers of artists performers and athletes'
  'agricultural and food scientists'
  'agricultural equipment operators'
  'agricultural sciences teachers postsecondary'
  'agricultural workers'
  'agricultural workers all other'
  'air traffic controllers and airfield operations specialists'
  'air transportation workers'
  'aircraft pilots and flight engineers'
  'aircraft structure surfaces rigging and systems assemblers'
  'anesthesiologists'
  'animal care and service workers'
  'anthropology and archeology teachers postsecondary'
  'arbitrators mediators and conciliators'
  'architects except landscape and naval'
  'architects except naval'
  'architects surveyors and cartographers'

Sample matched occupation names from GCSI table:
  'accountan

In [9]:
def normalize_name(s):
    """Normalize occupation name for fuzzy joining."""
    if pd.isna(s):
        return s
    s = str(s).lower().strip()
    s = s.replace(";", ",")        # semicolons → commas
    s = s.replace(" and ", ", ")   # 'and' → comma for grouped titles
    s = re.sub(r"\s+", " ", s)     # collapse extra spaces
    s = s.strip(",").strip()       # remove leading/trailing commas
    return s

import re

occupation_gcsi["name_key"] = occupation_gcsi["occupation_name"].apply(normalize_name)
best_match["career_key"]    = best_match["career_match"].apply(normalize_name)

best_enriched = best_match.merge(
    occupation_gcsi[["name_key", "SOC6", "GCSI",
                      "automation_probability", "A_MEDIAN", "TOT_EMP"]],
    left_on="career_key",
    right_on="name_key",
    how="left"
)

best_enriched["hybrid_score"] = (
    0.50 * best_enriched["similarity"] +
    0.50 * (best_enriched["GCSI"] / 100)
).round(4)

best_enriched.drop(columns=["name_key", "career_key"], inplace=True)
best_enriched.to_csv(f"{MAPPING}/program_best_enriched_gcsi.csv", index=False)

print(f"program_best_enriched_gcsi.csv saved: {best_enriched.shape}")
print(f"Rows with GCSI matched: {best_enriched['GCSI'].notna().sum()}")
print(f"Rows missing GCSI:      {best_enriched['GCSI'].isna().sum()}")

# Check remaining unmatched
still_unmatched = best_enriched[best_enriched["GCSI"].isna()]["career_match"].dropna().str.lower().str.strip().unique()
print(f"\nStill unmatched unique names: {len(still_unmatched)}")
for name in sorted(still_unmatched)[:15]:
    print(f"  '{name}'")

program_best_enriched_gcsi.csv saved: (57085, 15)
Rows with GCSI matched: 34622
Rows missing GCSI:      22463

Still unmatched unique names: 419
  'actors producers and directors'
  'administrative law judges adjudicators and hearing officers'
  'advertising marketing promotions public relations and sales managers'
  'agents and business managers of artists performers and athletes'
  'agricultural and food scientists'
  'agricultural equipment operators'
  'agricultural sciences teachers postsecondary'
  'agricultural workers'
  'agricultural workers all other'
  'air traffic controllers and airfield operations specialists'
  'air transportation workers'
  'aircraft pilots and flight engineers'
  'aircraft structure surfaces rigging and systems assemblers'
  'anesthesiologists'
  'animal care and service workers'


In [10]:
# How many of the 57085 program rows are affected?
affected_programs = best_enriched[best_enriched["GCSI"].isna()]["program_name"].nunique()
total_programs = best_enriched["program_name"].nunique()

print(f"Total unique programs:           {total_programs}")
print(f"Programs missing GCSI:           {affected_programs}")
print(f"Programs with GCSI:              {total_programs - affected_programs}")
print(f"Coverage:                        {((total_programs - affected_programs)/total_programs*100):.1f}%")

# What are the most common unmatched career names?
print("\nTop 20 most common unmatched career_match values:")
print(
    best_enriched[best_enriched["GCSI"].isna()]
    .groupby("career_match")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .to_string()
)

Total unique programs:           23780
Programs missing GCSI:           10407
Programs with GCSI:              13373
Coverage:                        56.2%

Top 20 most common unmatched career_match values:
career_match
biological scientists                                          1097
psychologists                                                   963
education administrators                                        844
foreign language and literature teachers postsecondary          612
legal occupations                                               557
english language and literature teachers postsecondary          529
engineers                                                       528
media and communication workers                                 480
business operations specialists                                 471
community health workers                                        421
management occupations                                          414
medical scientists              

In [11]:
from difflib import get_close_matches

# Build lookup: normalized GCSI name → original occupation name
gcsi_names = occupation_gcsi["occupation_name"].dropna().str.lower().str.strip().tolist()
gcsi_lookup = {n.lower().strip(): n for n in occupation_gcsi["occupation_name"].dropna()}

# For each unmatched career, find closest GCSI occupation
unmatched_list = (
    best_enriched[best_enriched["GCSI"].isna()]["career_match"]
    .dropna().str.lower().str.strip().unique().tolist()
)

fuzzy_map = {}
no_match  = []

for name in unmatched_list:
    matches = get_close_matches(name, gcsi_names, n=1, cutoff=0.6)
    if matches:
        fuzzy_map[name] = matches[0]
    else:
        no_match.append(name)

print(f"Fuzzy matched:     {len(fuzzy_map)}")
print(f"Still no match:    {len(no_match)}")

print("\nSample fuzzy mappings (unmatched → gcsi name):")
for k, v in list(fuzzy_map.items())[:20]:
    print(f"  '{k}'  →  '{v}'")

print("\nSample still unmatched:")
for name in no_match[:10]:
    print(f"  '{name}'")

Fuzzy matched:     363
Still no match:    56

Sample fuzzy mappings (unmatched → gcsi name):
  'business operations specialists'  →  'business operations specialists; all other'
  'engineers'  →  'ship engineers'
  'medical scientists'  →  'materials scientists'
  'physical scientists'  →  'political scientists'
  'education administrators'  →  'database administrators'
  'media and communication workers all other'  →  'production workers; all other'
  'art drama and music teachers postsecondary'  →  'education administrators; postsecondary'
  'miscellaneous life physical and social science technicians'  →  'life; physical; and social science technicians; all other'
  'obstetricians and gynecologists'  →  'food scientists and technologists'
  'personal care and service occupations'  →  'personal care aides'
  'miscellaneous health diagnosing and treating practitioners'  →  'health diagnosing and treating practitioners; all other'
  'clinical counseling and school psychologists'  →  'cl

In [12]:
# Manual mapping for the most impactful unmatched names
# Only mapping ones where we're confident about the right occupation
manual_map = {
    # broad → most representative specific occupation in GCSI table
    "engineers":                                          "engineers; all other",
    "psychologists":                                      "clinical; counseling; and school psychologists",
    "biological scientists":                              "biological scientists; all other",
    "education administrators":                           "education administrators; all other",
    "medical scientists":                                 "medical scientists; except epidemiologists",
    "social workers":                                     "social workers; all other",
    "writers and editors":                                "writers and authors",
    "electrical and electronics engineers":               "electrical engineers",
    "environmental scientists and geoscientists":         "environmental scientists and specialists; including health",
    "community health workers":                           "health educators",
    "clinical counseling and school psychologists":       "clinical; counseling; and school psychologists",
    "physical scientists":                                "physical scientists; all other",
    "business operations specialists":                    "business operations specialists; all other",
    "legal occupations":                                  "lawyers",
    "management occupations":                             "general and operations managers",
    "media and communication workers":                    "public relations specialists",
    "art drama and music teachers postsecondary":         "postsecondary teachers; all other",
    "foreign language and literature teachers postsecondary": "postsecondary teachers; all other",
    "english language and literature teachers postsecondary": "postsecondary teachers; all other",
    "philosophy and religion teachers postsecondary":     "postsecondary teachers; all other",
    "criminal justice and law enforcement teachers postsecondary": "postsecondary teachers; all other",
    "agricultural sciences teachers postsecondary":       "postsecondary teachers; all other",
    "anthropology and archeology teachers postsecondary": "postsecondary teachers; all other",
    "actors producers and directors":                     "producers and directors",
    "agricultural and food scientists":                   "food scientists and technologists",
    "agents and business managers of artists performers and athletes": "agents and business managers of artists; performers; and athletes",
    "administrative law judges adjudicators and hearing officers": "administrative law judges; adjudicators; and hearing officers",
    "advertising marketing promotions public relations and sales managers": "marketing managers",
    "agricultural equipment operators":                   "agricultural workers; all other",
    "agricultural workers":                               "agricultural workers; all other",
    "agricultural workers all other":                     "agricultural workers; all other",
    "air traffic controllers and airfield operations specialists": "air traffic controllers",
    "air transportation workers":                         "airline pilots; copilots; and flight engineers",
    "aircraft pilots and flight engineers":               "airline pilots; copilots; and flight engineers",
    "aircraft structure surfaces rigging and systems assemblers": "aircraft structure; surfaces; rigging; and systems assemblers",
    "anesthesiologists":                                  "anesthesiologists and anesthesiologist assistants",
    "animal care and service workers":                    "animal care and service workers; all other",
    "arbitrators mediators and conciliators":             "arbitrators; mediators; and conciliators",
    "architects except landscape and naval":              "architects; except landscape and naval",
    "nurse midwives":                                     "nurse midwives and nurse anesthetists",
    "preschool and kindergarten teachers":                "preschool teachers; except special education",
    "psychology teachers postsecondary":                  "postsecondary teachers; all other",
    "sales and related occupations":                      "sales representatives; wholesale and manufacturing",
    "secondary school teachers":                          "secondary school teachers; except special and career/technical education",
    "special education teachers":                         "special education teachers; all other",
    "arts design entertainment sports and media occupations": "art directors",
    "personal care and service occupations":              "personal care aides",
}

# Combine fuzzy (only safe ones) + manual
# We'll use manual_map only — skip fuzzy entirely since quality was poor
combined_map = manual_map.copy()

print(f"Manual mappings defined: {len(combined_map)}")

# Check which of our top unmatched are covered
top_unmatched = (
    best_enriched[best_enriched["GCSI"].isna()]
    .groupby("career_match")
    .size()
    .sort_values(ascending=False)
    .head(30)
)

print("\nTop 30 unmatched — covered by manual map?")
for name, count in top_unmatched.items():
    key = name.lower().strip()
    status = "✓" if key in combined_map else "✗ MISSING"
    print(f"  {status:10} {count:5}  '{name}'")

Manual mappings defined: 47

Top 30 unmatched — covered by manual map?
  ✓           1097  'biological scientists'
  ✓            963  'psychologists'
  ✓            844  'education administrators'
  ✓            612  'foreign language and literature teachers postsecondary'
  ✓            557  'legal occupations'
  ✓            529  'english language and literature teachers postsecondary'
  ✓            528  'engineers'
  ✓            480  'media and communication workers'
  ✓            471  'business operations specialists'
  ✓            421  'community health workers'
  ✓            414  'management occupations'
  ✓            408  'medical scientists'
  ✓            405  'art drama and music teachers postsecondary'
  ✓            375  'social workers'
  ✓            369  'electrical and electronics engineers'
  ✓            339  'philosophy and religion teachers postsecondary'
  ✓            329  'writers and editors'
  ✓            311  'criminal justice and law enforcement teach

In [13]:
# Add missing high-frequency ones
extra_map = {
    "media and communication workers all other":           "public relations specialists",
    "exercise physiologists":                              "exercise physiologists and physical therapists",
    "area ethnic and cultural studies teachers postsecondary": "postsecondary teachers; all other",
    "information security analysts":                       "information security analysts",
    "architecture and engineering occupations":            "architects; except landscape and naval",
    "software developers systems software":                "software developers",
    "art and design workers":                              "art directors",
}

combined_map.update(extra_map)
print(f"Total mappings: {len(combined_map)}")

# Now apply the full map
# Build a normalized name → SOC6+GCSI lookup from occupation_gcsi
gcsi_name_lookup = occupation_gcsi.copy()
gcsi_name_lookup["lookup_key"] = gcsi_name_lookup["occupation_name"].str.lower().str.strip()

def resolve_career(name):
    if pd.isna(name):
        return name
    key = str(name).lower().strip()
    return combined_map.get(key, key)  # return mapped name or original

# Apply mapping to best_match and redo the join
best_match["career_resolved"] = best_match["career_match"].apply(resolve_career)
best_match["career_resolved_key"] = best_match["career_resolved"].str.lower().str.strip()

best_enriched = best_match.merge(
    gcsi_name_lookup[["lookup_key", "SOC6", "GCSI",
                       "automation_probability", "A_MEDIAN", "TOT_EMP"]],
    left_on="career_resolved_key",
    right_on="lookup_key",
    how="left"
)

best_enriched["hybrid_score"] = (
    0.50 * best_enriched["similarity"] +
    0.50 * (best_enriched["GCSI"] / 100)
).round(4)

best_enriched.drop(columns=["lookup_key", "career_resolved_key"], inplace=True)
best_enriched.to_csv(f"{MAPPING}/program_best_enriched_gcsi.csv", index=False)

print(f"\nSaved: {best_enriched.shape}")
print(f"Rows with GCSI matched: {best_enriched['GCSI'].notna().sum()}")
print(f"Rows missing GCSI:      {best_enriched['GCSI'].isna().sum()}")

# Program-level coverage
covered = best_enriched[best_enriched["GCSI"].notna()]["program_name"].nunique()
total   = best_enriched["program_name"].nunique()
print(f"\nProgram coverage: {covered}/{total} = {covered/total*100:.1f}%")

Total mappings: 54

Saved: (57085, 17)
Rows with GCSI matched: 43144
Rows missing GCSI:      13941

Program coverage: 16237/23780 = 68.3%


In [14]:
# See what's still unmatched at high frequency
still_missing = (
    best_enriched[best_enriched["GCSI"].isna()]
    .groupby("career_match")
    .size()
    .sort_values(ascending=False)
    .head(30)
)
print("Still unmatched (top 30):")
print(still_missing.to_string())

Still unmatched (top 30):
career_match
education administrators                                       844
foreign language and literature teachers postsecondary         612
english language and literature teachers postsecondary         529
art drama and music teachers postsecondary                     405
social workers                                                 375
philosophy and religion teachers postsecondary                 339
criminal justice and law enforcement teachers postsecondary    311
special education teachers                                     287
exercise physiologists                                         286
area ethnic and cultural studies teachers postsecondary        282
information security analysts                                  260
software developers systems software                           220
fine artists including painters sculptors and illustrators     201
nurse practitioners                                            198
designers              

In [15]:
# Full combined map — all at once
full_map = {
    "biological scientists":                              "biological scientists; all other",
    "psychologists":                                      "clinical; counseling; and school psychologists",
    "education administrators":                           "education administrators; all other",
    "education administrators postsecondary":             "education administrators; postsecondary",
    "foreign language and literature teachers postsecondary": "postsecondary teachers; all other",
    "legal occupations":                                  "lawyers",
    "english language and literature teachers postsecondary": "postsecondary teachers; all other",
    "engineers":                                          "engineers; all other",
    "media and communication workers":                    "public relations specialists",
    "media and communication workers all other":          "public relations specialists",
    "business operations specialists":                    "business operations specialists; all other",
    "business operations specialists all other":          "business operations specialists; all other",
    "community health workers":                           "health educators",
    "management occupations":                             "general and operations managers",
    "medical scientists":                                 "medical scientists; except epidemiologists",
    "art drama and music teachers postsecondary":         "postsecondary teachers; all other",
    "social workers":                                     "social workers; all other",
    "electrical and electronics engineers":               "electrical engineers",
    "philosophy and religion teachers postsecondary":     "postsecondary teachers; all other",
    "writers and editors":                                "writers and authors",
    "criminal justice and law enforcement teachers postsecondary": "postsecondary teachers; all other",
    "environmental scientists and geoscientists":         "environmental scientists and specialists; including health",
    "clinical counseling and school psychologists":       "clinical; counseling; and school psychologists",
    "special education teachers":                         "special education teachers; all other",
    "special education teachers secondary school":        "special education teachers; secondary school",
    "exercise physiologists":                             "physical therapists",
    "area ethnic and cultural studies teachers postsecondary": "postsecondary teachers; all other",
    "information security analysts":                      "information security analysts",
    "architecture and engineering occupations":           "architects; except landscape and naval",
    "software developers systems software":               "software developers",
    "art and design workers":                             "art directors",
    "fine artists including painters sculptors and illustrators": "fine artists; including painters; sculptors; and illustrators",
    "nurse practitioners":                                "nurse practitioners",
    "designers":                                          "graphic designers",
    "chemists and materials scientists":                  "chemists",
    "other teachers and instructors":                     "postsecondary teachers; all other",
    "elementary and middle school teachers":              "elementary school teachers; except special education",
    "financial specialists":                              "financial analysts",
    "counselors all other":                               "counselors; all other",
    "conservation scientists and foresters":              "conservation scientists",
    "religious workers":                                  "clergy",
    "communications equipment operators":                 "telephone operators",
    "mathematical science teachers postsecondary":        "postsecondary teachers; all other",
    "computer network architects":                        "computer network architects",
    "biological science teachers postsecondary":          "postsecondary teachers; all other",
    "genetic counselors":                                 "genetic counselors",
    "actors producers and directors":                     "producers and directors",
    "agricultural and food scientists":                   "food scientists and technologists",
    "agents and business managers of artists performers and athletes": "agents and business managers of artists; performers; and athletes",
    "administrative law judges adjudicators and hearing officers": "administrative law judges; adjudicators; and hearing officers",
    "advertising marketing promotions public relations and sales managers": "marketing managers",
    "agricultural equipment operators":                   "agricultural workers; all other",
    "agricultural workers":                               "agricultural workers; all other",
    "agricultural workers all other":                     "agricultural workers; all other",
    "air traffic controllers and airfield operations specialists": "air traffic controllers",
    "air transportation workers":                         "airline pilots; copilots; and flight engineers",
    "aircraft pilots and flight engineers":               "airline pilots; copilots; and flight engineers",
    "aircraft structure surfaces rigging and systems assemblers": "aircraft structure; surfaces; rigging; and systems assemblers",
    "anesthesiologists":                                  "anesthesiologists and anesthesiologist assistants",
    "animal care and service workers":                    "animal care and service workers; all other",
    "arbitrators mediators and conciliators":             "arbitrators; mediators; and conciliators",
    "architects except landscape and naval":              "architects; except landscape and naval",
    "architects except naval":                            "architects; except landscape and naval",
    "architects surveyors and cartographers":             "architects; except landscape and naval",
    "nurse midwives":                                     "nurse midwives and nurse anesthetists",
    "preschool and kindergarten teachers":                "preschool teachers; except special education",
    "psychology teachers postsecondary":                  "postsecondary teachers; all other",
    "sales and related occupations":                      "sales representatives; wholesale and manufacturing",
    "secondary school teachers":                          "secondary school teachers; except special and career/technical education",
    "arts design entertainment sports and media occupations": "art directors",
    "personal care and service occupations":              "personal care aides",
    "physical scientists":                                "physical scientists; all other",
}

# Build clean lookup from GCSI table
gcsi_lookup = occupation_gcsi.copy()
gcsi_lookup["lookup_key"] = gcsi_lookup["occupation_name"].str.lower().str.strip()

# Apply map to best_match fresh (not on top of previous attempt)
bm = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_best_thr30.csv")

def resolve(name):
    if pd.isna(name):
        return name
    key = str(name).lower().strip()
    return full_map.get(key, key)

bm["career_key"] = bm["career_match"].apply(resolve)

# Join on resolved name
best_enriched = bm.merge(
    gcsi_lookup[["lookup_key", "SOC6", "GCSI",
                 "automation_probability", "A_MEDIAN", "TOT_EMP"]],
    left_on="career_key",
    right_on="lookup_key",
    how="left"
)

# Fill remaining missing GCSI with median (long-tail edge cases)
gcsi_median = best_enriched["GCSI"].median()
best_enriched["GCSI_imputed"] = best_enriched["GCSI"].isna()
best_enriched["GCSI"] = best_enriched["GCSI"].fillna(gcsi_median)

best_enriched["hybrid_score"] = (
    0.50 * best_enriched["similarity"] +
    0.50 * (best_enriched["GCSI"] / 100)
).round(4)

best_enriched.drop(columns=["lookup_key", "career_key"], inplace=True)
best_enriched.to_csv(f"{MAPPING}/program_best_enriched_gcsi.csv", index=False)

covered  = best_enriched[~best_enriched["GCSI_imputed"]]["program_name"].nunique()
total    = best_enriched["program_name"].nunique()
imputed  = best_enriched["GCSI_imputed"].sum()

print(f"Saved: {best_enriched.shape}")
print(f"Rows with real GCSI:    {(~best_enriched['GCSI_imputed']).sum()}")
print(f"Rows with imputed GCSI: {imputed}")
print(f"Program coverage (real GCSI): {covered}/{total} = {covered/total*100:.1f}%")
print(f"\nGCSI median used for imputation: {gcsi_median:.2f}")

Saved: (57085, 15)
Rows with real GCSI:    45120
Rows with imputed GCSI: 11965
Program coverage (real GCSI): 17259/23780 = 72.6%

GCSI median used for imputation: 41.63


In [16]:
# Preload top 5 skills per occupation by importance
top_skills = (
    skill_bridge
    .dropna(subset=["SOC6", "skill_name", "importance"])
    .sort_values("importance", ascending=False)
    .groupby("SOC6")
    .head(5)
    .groupby("SOC6")["skill_name"]
    .apply(list)
    .reset_index()
    .rename(columns={"skill_name": "top_skills"})
)

# Tech summary already loaded
top_tech = tech_summary[["SOC6", "technology_items", "hot_tech_items", "in_demand_items"]].copy()

# Make sure SOC6 types match for joining
top_skills["SOC6"] = top_skills["SOC6"].astype(str).str.strip()
top_tech["SOC6"]   = top_tech["SOC6"].astype(str).str.strip()
best_enriched["SOC6"] = best_enriched["SOC6"].astype(str).str.strip()

def recommend(degree_name, top_n=10):
    """
    Input:  degree name (partial match ok, case insensitive)
    Output: top N careers ranked by hybrid score
            with GCSI breakdown, salary, skills, tech tools
    """
    mask = best_enriched["degree_final"].str.lower().str.contains(
        degree_name.lower().strip(), na=False
    )
    results = best_enriched[mask].copy()

    if results.empty:
        print(f"No matches found for: '{degree_name}'")
        return pd.DataFrame()

    # Sort by hybrid score, take top N
    results = (
        results
        .sort_values("hybrid_score", ascending=False)
        .drop_duplicates(subset=["career_match"])
        .head(top_n)
    )

    # Attach skills
    results = results.merge(top_skills, on="SOC6", how="left")

    # Attach tech
    results = results.merge(top_tech, on="SOC6", how="left")

    display_cols = [
        "program_name", "program_type", "career_match",
        "similarity", "GCSI", "GCSI_imputed", "hybrid_score",
        "A_MEDIAN", "automation_probability",
        "top_skills", "technology_items", "hot_tech_items"
    ]

    return results[display_cols].reset_index(drop=True)


# Test
print("=== computer science ===")
cs = recommend("computer science", top_n=5)
print(cs[["career_match", "similarity", "GCSI", "hybrid_score", "A_MEDIAN"]].to_string())

print("\n=== economics ===")
econ = recommend("economics", top_n=5)
print(econ[["career_match", "similarity", "GCSI", "hybrid_score", "A_MEDIAN"]].to_string())

print("\n=== business administration ===")
biz = recommend("business administration", top_n=5)
print(biz[["career_match", "similarity", "GCSI", "hybrid_score", "A_MEDIAN"]].to_string())

=== computer science ===
                                   career_match  similarity   GCSI  hybrid_score  A_MEDIAN
0     computer and information systems managers      0.6906  61.91        0.6548  135800.0
1          electrical and electronics engineers      0.8521  45.03        0.6512   94210.0
2  computer and information research scientists      0.7693  47.74        0.6234  111840.0
3                   computer hardware engineers      0.7977  44.54        0.6216  115080.0
4                     computer systems analysts      0.7277  48.11        0.6044   87220.0

=== economics ===
                                career_match  similarity   GCSI  hybrid_score  A_MEDIAN
0                         financial managers      0.7651  59.25        0.6788  121750.0
1  computer and information systems managers      0.6625  61.91        0.6408  135800.0
2                         marketing managers      0.6415  60.52        0.6234  131180.0
3           economics teachers postsecondary      0.8100  

In [17]:
print("=== GCSI top 10 careers ===")
print(occupation_gcsi[["occupation_name", "GCSI"]].sort_values("GCSI", ascending=False).head(10).to_string())

print("\n=== GCSI bottom 10 careers ===")
print(occupation_gcsi[["occupation_name", "GCSI"]].sort_values("GCSI").head(10).to_string())

print("\n=== Imputed vs real GCSI distribution ===")
print(best_enriched.groupby("GCSI_imputed")["hybrid_score"].describe().round(3))

print("\n=== Test: nursing ===")
nur = recommend("nursing", top_n=5)
print(nur[["career_match", "GCSI", "hybrid_score", "A_MEDIAN", "top_skills"]].to_string())

print("\n=== Test: public administration ===")
pub = recommend("public administration", top_n=5)
print(pub[["career_match", "GCSI", "hybrid_score", "A_MEDIAN"]].to_string())

=== GCSI top 10 careers ===
                               occupation_name   GCSI
0                             Chief Executives  71.41
232                          Dentists; General  65.07
7    Computer and Information Systems Managers  61.91
1              General and Operations Managers  61.66
3                           Marketing Managers  60.52
165                                    Lawyers  59.57
8                           Financial Managers  59.25
4                               Sales Managers  58.56
241                                Podiatrists  58.25
235                            Prosthodontists  57.46

=== GCSI bottom 10 careers ===
                                   occupation_name   GCSI
377                            Real Estate Brokers  12.65
39                          Farm Labor Contractors  15.37
210  Umpires; Referees; and Other Sports Officials  17.36
376                                         Models  17.60
59                                   Tax Preparers  17.7

In [18]:
# Check why nursing has no skills
nursing_careers = ["nurse anesthetists", "nurse practitioners", 
                   "registered nurses", "nurse midwives"]

print("=== SOC6 for nursing in best_enriched ===")
print(best_enriched[best_enriched["career_match"].isin(nursing_careers)][["career_match", "SOC6", "GCSI", "A_MEDIAN"]].drop_duplicates())

print("\n=== SOC6 for nursing in occupation_gcsi ===")
print(occupation_gcsi[occupation_gcsi["occupation_name"].str.contains("nurse|Nurse", na=False)][["SOC6", "occupation_name", "GCSI", "A_MEDIAN"]])

print("\n=== SOC6 for nursing in skill_bridge ===")
print(skill_bridge[skill_bridge["occupation_name"].str.contains("nurse|Nurse", na=False)][["SOC6", "occupation_name"]].drop_duplicates())

print("\n=== SOC6 types ===")
print(f"best_enriched SOC6 dtype:  {best_enriched['SOC6'].dtype}")
print(f"top_skills SOC6 dtype:     {top_skills['SOC6'].dtype}")
print(f"occupation_gcsi SOC6 dtype: {occupation_gcsi['SOC6'].dtype}")

=== SOC6 for nursing in best_enriched ===
            career_match     SOC6   GCSI  A_MEDIAN
39        nurse midwives      NaN  41.63       NaN
40     registered nurses  29-1111  40.98       NaN
131   nurse anesthetists      NaN  41.63       NaN
293  nurse practitioners      NaN  41.63       NaN

=== SOC6 for nursing in occupation_gcsi ===
        SOC6                                    occupation_name   GCSI  \
242  29-1111                                  Registered Nurses  40.98   
266  29-2061  Licensed Practical and Licensed Vocational Nurses  45.61   

     A_MEDIAN  
242       NaN  
266   44090.0  

=== SOC6 for nursing in skill_bridge ===
         SOC6                                    occupation_name
7910  29-2061  Licensed Practical and Licensed Vocational Nurses

=== SOC6 types ===
best_enriched SOC6 dtype:  str
top_skills SOC6 dtype:     str
occupation_gcsi SOC6 dtype: str


In [19]:
# Add to full_map before the resolve step
nursing_additions = {
    "nurse anesthetists":                    "registered nurses",
    "nurse practitioners":                   "registered nurses", 
    "nurse midwives":                        "registered nurses",
    "nurse midwives and nurse anesthetists": "registered nurses",
}
full_map.update(nursing_additions)

# Also fix A_MEDIAN NaN — fill from annual median where hourly is missing
# Check salary file for registered nurses
print("Salary rows for 29-1111:")
print(salary[salary["OCC_CODE"].astype(str).str.strip() == "29-1111"][["OCC_CODE", "OCC_TITLE", "OCC_GROUP", "TOT_EMP", "A_MEDIAN", "H_MEDIAN"]])

print("\nSalary rows containing 'nurse':")
print(salary[salary["OCC_TITLE"].str.contains("Nurse|nurse", na=False)][["OCC_CODE", "OCC_TITLE", "OCC_GROUP", "A_MEDIAN"]].to_string())

Salary rows for 29-1111:
Empty DataFrame
Columns: [OCC_CODE, OCC_TITLE, OCC_GROUP, TOT_EMP, A_MEDIAN, H_MEDIAN]
Index: []

Salary rows containing 'nurse':
    OCC_CODE                                                OCC_TITLE OCC_GROUP A_MEDIAN
533  29-1140                                        Registered Nurses     broad    68450
534  29-1141                                        Registered Nurses  detailed    68450
535  29-1150                                       Nurse Anesthetists     broad   160270
536  29-1151                                       Nurse Anesthetists  detailed   160270
537  29-1160                                           Nurse Midwives     broad    99770
538  29-1161                                           Nurse Midwives  detailed    99770
539  29-1170                                      Nurse Practitioners     broad   100910
540  29-1171                                      Nurse Practitioners  detailed   100910
567  29-2060        Licensed Practical and L

In [2]:
import pandas as pd
df = pd.read_csv("datasets/program_best_enriched_gcsi.csv")
aero = df[df["program_name"] == "Aerospace Systems Engineering"]
print(aero[["program_name", "program_type", "career_match", "similarity", "GCSI"]].to_string())

                        program_name program_type         career_match  similarity   GCSI
40864  Aerospace Systems Engineering          MSc  aerospace engineers      0.8319  49.16
43178  Aerospace Systems Engineering          MSc  aerospace engineers      0.8319  49.16


In [3]:
import pandas as pd

MAPPING = "../Progress/code/datasets/mapping"

# Check what's in each file
best   = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_best_thr30.csv")
cands  = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_candidates_thr30_long.csv")

print("=== best_match table ===")
print(f"Shape: {best.shape}")
print(f"Columns: {best.columns.tolist()}")
print(f"Unique programs: {best['program_name'].nunique()}")
print(f"Rows per program (should be 1): {best.groupby('program_name').size().describe()}")

print("\n=== candidates_long table ===")
print(f"Shape: {cands.shape}")
print(f"Columns: {cands.columns.tolist()}")
print(f"Unique programs: {cands['program_name'].nunique()}")
print(f"Rows per program: {cands.groupby('program_name').size().describe()}")

print("\n=== Sample: all candidates for Economics MSc ===")
econ = cands[(cands['program_name'] == 'Economics') & 
             (cands['program_type'] == 'MSc')]
print(econ[['program_name', 'program_type', 'career_match', 'similarity']].head(20).to_string())

=== best_match table ===
Shape: (57085, 8)
Columns: ['program_row_id', 'program_name', 'program_type', 'degree_source', 'degree_final', 'match_rank', 'career_match', 'similarity']
Unique programs: 23780
Rows per program (should be 1): count    23780.000000
mean         2.400547
std         11.044081
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        427.000000
dtype: float64

=== candidates_long table ===
Shape: (4357770, 8)
Columns: ['program_row_id', 'program_name', 'program_type', 'degree_source', 'degree_final', 'match_rank', 'career_match', 'similarity']
Unique programs: 23780
Rows per program: count     23780.000000
mean        183.253574
std        1759.796572
min           1.000000
25%          17.000000
50%          43.000000
75%          97.000000
max      113391.000000
dtype: float64

=== Sample: all candidates for Economics MSc ===
   program_name program_type                                        career_match  similarity
0  